# `<TICKER>__Layer1_6_to_Layer4_2.ipynb` — per-asset runner

Runs Layers **1.6 → 4.2** for one ticker and writes the seven-file deliverable into `Assets/<TICKER>/`.
`run_asset.py` injects the ticker into the setup cell, executes this notebook, and saves the executed copy as
file #1. Compute lives in `pipeline.py`; the three file-writers live in `asset_writers.py`.

## 00 — Setup

In [ ]:
import sys
from pathlib import Path

STRUCTURE_DIR = str(Path.cwd())          # operational root (run from Project/Structure/)
sys.path.insert(0, STRUCTURE_DIR)
import pipeline as P
import asset_writers as W

TICKER = "AAPL"                          # run_asset.py overrides this line per ticker
DB_PATH = str(Path(STRUCTURE_DIR) / "liora.duckdb")

ASSET_DIR = Path(STRUCTURE_DIR) / "Assets" / TICKER
ASSET_DIR.mkdir(parents=True, exist_ok=True)
AF = {
    "notebook":   ASSET_DIR / f"{TICKER}__Layer1_6_to_Layer4_2.ipynb",
    "parquet":    ASSET_DIR / f"{TICKER}_ohlcv_1h.parquet",
    "parquet_1d": ASSET_DIR / f"{TICKER}_ohlcv_1d.parquet",
    "parquet_1w": ASSET_DIR / f"{TICKER}_ohlcv_1w.parquet",
    "optuna":     ASSET_DIR / "OPTUNAs_XGB_HPOs_best_params.json",
    "strategy":   ASSET_DIR / f"strategy_{TICKER}.py",
    "readme":     ASSET_DIR / f"{TICKER}_README.md",
}
P.validate_parameters()
SEED = P.seed_everything()
MANIFEST = P.resolve_feature_manifest(TICKER)
THRESH = P.PIPELINE_PARAMETERS["THRESHOLD_ENTRY"]
print("TICKER", TICKER, "| DB", DB_PATH, "| features", MANIFEST["effective_feature_count"])

## 1.6 — DuckDB snapshot → clean 1h parquet, then roll up to 1d / 1w

In [ ]:
# Layer 1.6 — read bars_1h from liora.duckdb, assert the clean-OHLCV contract, write file #2
df = P.layer1_6_snapshot_to_parquet(DB_PATH, TICKER, AF["parquet"])
print("1.6 clean 1h rows:", len(df), "|", df["timestamp"].min(), "->", df["timestamp"].max())

In [ ]:
# Layer 1.6 — deterministic 1h -> 1d (ET day) / 1w (ISO week) roll-up, writes files #3 and #4
P.layer1_6_materialize_timeframes(df, {"1d": AF["parquet_1d"], "1w": AF["parquet_1w"]})
import pandas as pd
for tf in ("1d", "1w"):
    d = pd.read_parquet(AF[f"parquet_{tf}"])
    print(f"1.6 {tf} rows:", len(d), "| cols:", list(d.columns))

## 1.7 → 2.3 — time split, setup detector, features X + label Y

In [ ]:
# Layers 1.7 split -> 2.1 detect -> purge -> 2.3 resolve -> 2.2 Output B (all in memory)
REC = P.derive_output_b(df, TICKER, MANIFEST)
print("train setups:", len(REC["train_setups"]), "| Output-B rows:", len(REC["df_b"]))
print("bounds:", REC["bounds"])

## 3.1 — Optuna best hyperparameters

In [ ]:
# Layer 3.1 — Optuna TPE + MedianPruner over purged walk-forward CV (maximize AUC-PR on Train)
BEST, AP, FOLDS = P.layer3_1_optuna(REC["df_b"], REC["bounds"], SEED, MANIFEST)
print("best:", BEST)
print("cv AUC-PR:", AP, "| folds:", FOLDS)

In [ ]:
# Write file #5 — OPTUNAs_XGB_HPOs_best_params.json (best params + resolved manifest + CV diagnostics)
import json
json.dump({"best_params": BEST, "feature_manifest": MANIFEST, "cv_auc_pr": AP, "cv_folds": FOLDS,
           "n_trials": P.n_trials()}, open(AF["optuna"], "w"), indent=2)
print("wrote", AF["optuna"].name)

## 3.2 — XGBoost strategy artifact

In [ ]:
# Layer 3.2 — train XGBoost on full Train; score Train + Train acceptance via the shared engine
BOOSTER = P.layer3_2_train(REC["df_b"], BEST, SEED, MANIFEST)
tr_scored = P.score_setups(BOOSTER, REC["df"], REC["train_setups"], MANIFEST, REC["context_states"])
tr_summary, _, _ = P.run_engine(REC["df"], tr_scored, REC["bounds"]["train_start_idx"],
                                REC["bounds"]["oos_start_idx"] - 1, THRESH)
ACCEPT = P.accept_strategy(tr_summary)
print("train acceptance:", ACCEPT)

In [ ]:
# Write file #6 — strategy_<TICKER>.py (standalone artifact: base64 model + selfcheck)
sp = P.PIPELINE_PARAMETERS["splits"]
META = P.strategy_meta(BOOSTER, REC["df_b"], TICKER, BEST, {},
                       {"start": sp["train_start"], "end": sp["train_end"]}, MANIFEST)
META["ACCEPTANCE"] = ACCEPT
W.write_strategy(AF["strategy"], META)
print("wrote", AF["strategy"].name)

## 4.1 — OOS verdict

In [ ]:
# Layer 4.1 — one-shot OOS on the in-RAM booster (same shared run_engine as Train acceptance)
oos_setups, _ = P.layer2_1_detect(df, REC["masks"]["oos"])
oos_scored = P.score_setups(BOOSTER, df, oos_setups, MANIFEST, REC["context_states"])
SUMM, LEDGER, _ = P.run_engine(df, oos_scored, REC["bounds"]["oos_start_idx"],
                               REC["bounds"]["oos_end_idx"], THRESH)
print("OOS:", {k: SUMM[k] for k in ("start_capital", "end_capital", "profit_factor", "trades")})

## 4.2 — README + final 7-file check

In [ ]:
# Write file #7 — <TICKER>_README.md (OOS capital path + feature table + Risk-Box ledger)
W.write_readme(AF["readme"], {**SUMM, "ticker": TICKER}, LEDGER, MANIFEST)
print("wrote", AF["readme"].name)

In [ ]:
# Layer 4.1 results store — UPSERT this asset's OOS verdict into oos_metrics.db (the per-asset table the Dashboard reads).
# Side-effect OUTSIDE the 7 files: lives in Structure/, not Assets/ — the 7-file contract is unaffected.
OOS_DB = Path(STRUCTURE_DIR) / "oos_metrics.db"
W.write_oos_metrics(OOS_DB, {**SUMM, "ticker": TICKER, "cv_auc_pr": AP, "cv_folds": FOLDS,
                             "oos_window": f'{sp["oos_start"]} → {sp["oos_end"]}'})
print("4.1 →", OOS_DB.name, ":", TICKER)

In [ ]:
# Deliverable check. File #1 (this notebook's executed copy) is written by run_asset.py after this finishes,
# so we verify the other six here; run_asset.py does the authoritative 7-file check.
six = [AF["parquet"], AF["parquet_1d"], AF["parquet_1w"], AF["optuna"], AF["strategy"], AF["readme"]]
missing = [p.name for p in six if not p.exists()]
assert not missing, f"missing deliverables: {missing}"
print("6/7 deliverables present (notebook copy added by run_asset.py):")
print(sorted(p.name for p in ASSET_DIR.iterdir() if p.is_file()))